# advanced_sequence_adapter_batch_v1

验证第二批 sequence/deep adapter：`alstm_mse`、`sfm_mse`、`darnn_mse`、`tcn_mse`。本轮目标是把注意力、频域和 temporal convolution 三类输入机制接入统一 `src/model` 训练、预测、artifact 和 matrix analysis 链路。

## 实验治理

- Baseline：上一轮完成 `sequence_lag_ridge_mse`、`lstm_mse`、`gru_mse`，本轮扩展到注意力/频域/卷积结构。
- Direct comparison：ALSTM attention LSTM、SFM frequency proxy、DA-RNN dual attention proxy、TCN temporal conv 是否输出不同预测并保持同一 artifact/manifest 契约。
- Role：diagnose，推进 QuantBench sequence family 从 smoke fallback 迁移到可训练 adapter。
- Target window：2026-01-01 到 2026-06-09，2026-04-30 前训练，之后测试。
- Changed layer：model。
- External reference：DA-RNN 双阶段注意力时间序列预测、TCN sequence modeling、PyTorch recurrent/Conv1d module 文档。SFM 当前实现为频域特征 proxy，不宣称完整 SFM 论文结构。
- Acceptance：四个模型输出 `paper_strict_adapter`；attention/frequency/tcn 三类路径预测互不相同；DA-RNN artifact 保存/加载后保留 adapter metadata。

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

repo_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'src').exists() and (path / 'AGENTS.md').exists()
)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.data import build_supervised_dataset, load_soxl_support_frame
from src.model import fit_model_artifact, load_model_artifact, save_model_artifact
from src.model.matrix import run_quantbench_matrix_smoke
from src.model.training import fit_predict_model, model_adapter_name, model_adapter_status

CARD = repo_root / 'research' / 'advanced_sequence_adapter_batch_v1'
CARD.mkdir(parents=True, exist_ok=True)

frame = load_soxl_support_frame(start='2026-01-01', end='2026-06-09')
dataset = build_supervised_dataset(frame, horizon=1)
train = dataset.frame.loc[dataset.frame['date'] <= '2026-04-30'].copy()
test = dataset.frame.loc[dataset.frame['date'] > '2026-04-30'].copy()
feature_columns = dataset.feature_columns[:5]
{
    'train_rows': int(len(train)),
    'test_rows': int(len(test)),
    'feature_columns': list(feature_columns),
    'train_end': str(train['date'].max().date()),
    'test_start': str(test['date'].min().date()),
    'test_end': str(test['date'].max().date()),
}


{'train_rows': 82,
 'test_rows': 26,
 'feature_columns': ['soxl_return_1d',
  'soxx_return_1d',
  'soxl_return_5d',
  'soxx_return_5d',
  'soxl_soxx_relative_5d'],
 'train_end': '2026-04-30',
 'test_start': '2026-05-01',
 'test_end': '2026-06-08'}

In [2]:
model_names = ('alstm_mse', 'sfm_mse', 'darnn_mse', 'tcn_mse')
expected = {
    'alstm_mse': ('paper_strict_adapter', 'alstm_attention_lstm_mse_v1'),
    'sfm_mse': ('paper_strict_adapter', 'sfm_frequency_ridge_mse_v1'),
    'darnn_mse': ('paper_strict_adapter', 'darnn_dual_attention_gru_mse_v1'),
    'tcn_mse': ('paper_strict_adapter', 'tcn_temporal_conv_mse_v1'),
}

predictions = pd.concat(
    [fit_predict_model(name, train=train, test=test, feature_columns=feature_columns) for name in model_names],
    ignore_index=True,
)
predictions.to_csv(CARD / 'adapter_predictions.csv', index=False)

for name, (status, adapter_name) in expected.items():
    subset = predictions.loc[predictions['model'] == name]
    assert subset['adapter_status'].unique().tolist() == [status]
    assert subset['model_adapter'].unique().tolist() == [adapter_name]
    assert model_adapter_status(name) == status
    assert model_adapter_name(name) == adapter_name

wide = predictions.pivot(index='date', columns='model', values='prediction').reset_index()
wide['alstm_minus_sfm'] = wide['alstm_mse'] - wide['sfm_mse']
wide['darnn_minus_alstm'] = wide['darnn_mse'] - wide['alstm_mse']
wide['tcn_minus_sfm'] = wide['tcn_mse'] - wide['sfm_mse']
wide.to_csv(CARD / 'advanced_sequence_prediction_deltas.csv', index=False)

assert not wide['alstm_mse'].round(10).equals(wide['sfm_mse'].round(10))
assert not wide['darnn_mse'].round(10).equals(wide['alstm_mse'].round(10))
assert not wide['tcn_mse'].round(10).equals(wide['sfm_mse'].round(10))
wide.head()


model,date,alstm_mse,darnn_mse,sfm_mse,tcn_mse,alstm_minus_sfm,darnn_minus_alstm,tcn_minus_sfm
0,2026-05-01,-0.000334,0.001002,-0.028424,0.002397,0.028090,0.001335,0.030821
1,2026-05-04,0.001033,0.000345,-0.020784,0.007603,0.021817,-0.000688,0.028387
2,2026-05-05,0.003633,0.001849,0.033724,0.032824,-0.030091,-0.001785,-0.000900
3,2026-05-06,0.020538,0.005768,0.060494,0.035581,-0.039956,-0.014770,-0.024913
4,2026-05-07,0.027212,0.010949,0.040873,0.031474,-0.013661,-0.016264,-0.009399


In [3]:
summary_rows = []
group_columns = ['model', 'objective', 'family', 'adapter_status', 'model_adapter']
for keys, group in predictions.groupby(group_columns, dropna=False):
    summary_rows.append(
        {
            **dict(zip(group_columns, keys)),
            'rows': int(len(group)),
            'prediction_mean': float(group['prediction'].mean()),
            'prediction_std': float(group['prediction'].std(ddof=0)),
            'target_corr': float(group['prediction'].corr(group['target_return'])),
        }
    )
adapter_summary = pd.DataFrame(summary_rows)
adapter_summary.to_csv(CARD / 'adapter_summary.csv', index=False)
adapter_summary


,model,objective,family,adapter_status,model_adapter,rows,prediction_mean,prediction_std,target_corr
0,alstm_mse,mse,sequence,paper_strict_adapter,alstm_attention_lstm_mse_v1,26,0.027433,0.014545,-0.321971
1,darnn_mse,mse,sequence,paper_strict_adapter,darnn_dual_attention_gru_mse_v1,26,0.025580,0.013986,-0.251606
2,sfm_mse,mse,sequence,paper_strict_adapter,sfm_frequency_ridge_mse_v1,26,0.034486,0.026187,0.030699
3,tcn_mse,mse,sequence,paper_strict_adapter,tcn_temporal_conv_mse_v1,26,0.025111,0.011649,-0.195045


In [4]:
artifact = fit_model_artifact('darnn_mse', train=train, feature_columns=feature_columns)
artifact_path = save_model_artifact(CARD / 'darnn_mse_artifact.joblib', artifact)
loaded = load_model_artifact(artifact_path)
artifact_predictions = loaded.predict(test)
artifact_metadata = loaded.metadata_payload()
artifact_predictions.to_csv(CARD / 'darnn_artifact_predictions.csv', index=False)
(CARD / 'darnn_artifact_metadata.json').write_text(
    json.dumps(artifact_metadata, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

assert loaded.adapter_status == 'paper_strict_adapter'
assert artifact_metadata['model_adapter'] == 'darnn_dual_attention_gru_mse_v1'
assert artifact_predictions['model_adapter'].unique().tolist() == ['darnn_dual_attention_gru_mse_v1']
{
    'artifact_path': str(artifact_path.relative_to(CARD)),
    'metadata': artifact_metadata,
    'prediction_rows': int(len(artifact_predictions)),
}


{'artifact_path': 'darnn_mse_artifact.joblib',
 'metadata': {'schema_version': 'model_artifact.v1',
  'model_name': 'darnn_mse',
  'feature_columns': ['soxl_return_1d',
   'soxx_return_1d',
   'soxl_return_5d',
   'soxx_return_5d',
   'soxl_soxx_relative_5d'],
  'objective': 'mse',
  'family': 'sequence',
  'adapter_status': 'paper_strict_adapter',
  'model_adapter': 'darnn_dual_attention_gru_mse_v1',
  'target_column': 'target_return',
  'train_rows': 82},
 'prediction_rows': 26}

In [5]:
matrix = run_quantbench_matrix_smoke(
    card_dir=CARD / 'matrix_card',
    start='2026-01-01',
    end='2026-06-09',
    train_end='2026-04-30',
    model_names=model_names,
)
manifest = json.loads((CARD / 'matrix_card' / 'artifact_manifest.json').read_text(encoding='utf-8'))
assert manifest['metadata']['adapter_status'] == 'paper_strict_adapter'
assert manifest['metadata']['adapter_statuses'] == ['paper_strict_adapter']
matrix.model_comparison.to_csv(CARD / 'matrix_model_comparison.csv', index=False)
{
    'matrix_status': matrix.summary['status'],
    'top_model': matrix.summary['top_model'],
    'adapter_status': manifest['metadata']['adapter_status'],
    'model_adapters': manifest['metadata']['model_adapters'],
    'manifest_hash': manifest['manifest_hash'],
}


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'matrix_status': 'smoke_completed',
 'top_model': 'darnn_mse',
 'adapter_status': 'paper_strict_adapter',
 'model_adapters': ['alstm_attention_lstm_mse_v1',
  'darnn_dual_attention_gru_mse_v1',
  'sfm_frequency_ridge_mse_v1',
  'tcn_temporal_conv_mse_v1'],
 'manifest_hash': 'ab94084049255a2642417708fc037caabcc38c3f462c4bc3842007ca8a9c4ee2'}

In [6]:
analysis_summary = {
    'schema_version': 'advanced_sequence_adapter_batch_review.v1',
    'status': 'completed',
    'train_rows': int(len(train)),
    'test_rows': int(len(test)),
    'feature_columns': list(feature_columns),
    'models': list(model_names),
    'adapter_statuses': sorted(predictions['adapter_status'].unique().tolist()),
    'model_adapters': sorted(predictions['model_adapter'].unique().tolist()),
    'alstm_vs_sfm_abs_mean_diff': float(wide['alstm_minus_sfm'].abs().mean()),
    'darnn_vs_alstm_abs_mean_diff': float(wide['darnn_minus_alstm'].abs().mean()),
    'tcn_vs_sfm_abs_mean_diff': float(wide['tcn_minus_sfm'].abs().mean()),
    'adapter_summary_path': 'adapter_summary.csv',
    'prediction_path': 'adapter_predictions.csv',
    'delta_path': 'advanced_sequence_prediction_deltas.csv',
    'artifact_path': 'darnn_mse_artifact.joblib',
    'matrix_manifest': 'matrix_card/artifact_manifest.json',
    'matrix_top_model': matrix.summary['top_model'],
    'matrix_manifest_hash': manifest['manifest_hash'],
    'external_references': [
        'https://arxiv.org/abs/1704.02971',
        'https://arxiv.org/abs/1803.01271',
        'https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html',
        'https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv1d.html',
    ],
}
(CARD / 'analysis_summary.json').write_text(json.dumps(analysis_summary, ensure_ascii=False, indent=2), encoding='utf-8')
analysis_summary


{'schema_version': 'advanced_sequence_adapter_batch_review.v1',
 'status': 'completed',
 'train_rows': 82,
 'test_rows': 26,
 'feature_columns': ['soxl_return_1d',
  'soxx_return_1d',
  'soxl_return_5d',
  'soxx_return_5d',
  'soxl_soxx_relative_5d'],
 'models': ['alstm_mse', 'sfm_mse', 'darnn_mse', 'tcn_mse'],
 'adapter_statuses': ['paper_strict_adapter'],
 'model_adapters': ['alstm_attention_lstm_mse_v1',
  'darnn_dual_attention_gru_mse_v1',
  'sfm_frequency_ridge_mse_v1',
  'tcn_temporal_conv_mse_v1'],
 'alstm_vs_sfm_abs_mean_diff': 0.024609563432988597,
 'darnn_vs_alstm_abs_mean_diff': 0.0072039862854074216,
 'tcn_vs_sfm_abs_mean_diff': 0.02185578306094494,
 'adapter_summary_path': 'adapter_summary.csv',
 'prediction_path': 'adapter_predictions.csv',
 'delta_path': 'advanced_sequence_prediction_deltas.csv',
 'artifact_path': 'darnn_mse_artifact.joblib',
 'matrix_manifest': 'matrix_card/artifact_manifest.json',
 'matrix_top_model': 'darnn_mse',
 'matrix_manifest_hash': 'ab9408404925

## 中文结论

- 做了什么：新增并验证 `alstm_mse`、`sfm_mse`、`darnn_mse`、`tcn_mse` 四个 sequence/deep adapter；其中 ALSTM/DA-RNN 使用注意力 recurrent proxy，SFM 使用频域特征 proxy，TCN 使用 temporal convolution proxy。
- 核心指标：四个模型的 adapter 状态、预测均值/波动、预测与目标相关性写入 `adapter_summary.csv`；ALSTM vs SFM、DA-RNN vs ALSTM、TCN vs SFM 的平均绝对预测差异写入 `analysis_summary.json`。
- regime 差异：本轮小矩阵通过 `matrix_card` 写出 `regime_metrics.csv`、`window_metrics.csv`、`qlib_risk_metrics.csv` 和 `model_effect_deltas.csv`，证明高级 sequence adapter 能进入已有 qlib/regime 分析链路。
- 是否晋升：否。本轮是 adapter 框架推进，不代表生产策略晋升。
- 风险：这些仍是最小可训练 proxy，尤其 SFM 和 DA-RNN 还不是完整论文级结构；后续要用更严格的论文复现替换同名 adapter 内部实现。
- 下一步：继续接入 `hawkes_gru_mse`、`mlp_mixer_mse`、`dlinear_mse`、`patch_tst_mse`、`autoformer_mse`、`fedformer_mse`，并保持同一 artifact/manifest/dashboard/analysis 契约。